# StyleMe — 01: Taxonomy, Zero-shot Eval & Model Export

Builds the vision pipeline and validates it against **real labels** (not vibes).

| attribute | eval set | gate |
|---|---|---|
| slot (6 classes) | DeepFashion2 val bbox + category | >= 0.90 |
| pattern | Fashionpedia textile-pattern attributes | >= 0.80 |
| fit | Fashionpedia silhouette-fit attributes | >= 0.75 |
| color (classical CV) | Fashion Product `baseColour` | >= 0.85 |
| formality | Fashion Product `usage` (coarse) | >= 0.70 exact |
| material / season | visual grid | human check |

**Model:** `MobileCLIP-S1` (FastViT-MCi1, 256px, 512-d, ~21M params). S0 is not in `open_clip`; S1 is the smallest supported and still mobile-sized. TFLite fp16 export is ~40 MB.

**Outputs** (`artifacts/`):
- `prompt_embeddings.npy` + `taxonomy.json` — prompt vectors shipped in the app
- `mobileclip_s1_image.onnx` + `mobileclip_s1_image.tflite` (fp16) — on-device encoder
- `probe_data.npz` — embeddings + labels for notebook 02
- `eval_report.md` / `eval_report.json`
- `styleme_vision_v1.zip` — app bundle

If a gate fails, run **02_probe_finetune_and_export.ipynb** (linear probes on frozen embeddings), then re-bundle.

In [ ]:
!pip install -q open-clip-torch pyyaml opencv-python-headless scikit-learn pandas pyarrow pillow huggingface_hub onnx onnx2tf

In [ ]:
import os, io, json, glob, zipfile, subprocess, warnings, ast, shutil
from collections import Counter
import numpy as np
import pandas as pd

ROOT = os.environ.get('STYLEME_ROOT')
if not ROOT:
    try:
        from google.colab import drive
        drive.mount('/content/drive', timeout_notify=20)
        ROOT = '/content/drive/MyDrive/StyleMe'
    except Exception:
        ROOT = '/content/StyleMe'

RAW = os.path.join(ROOT, 'raw')
ART = os.path.join(ROOT, 'artifacts')
os.makedirs(ART, exist_ok=True)
print('ROOT:', ROOT)
for d in sorted(os.listdir(RAW)):
    print(' raw/', d)

In [ ]:
import yaml
tax_path = next(iter(filter(os.path.exists, [
    os.path.join(ROOT, 'taxonomy.yaml'),
    '/content/ml/taxonomy.yaml',
    '/content/taxonomy.yaml',
])), None)
assert tax_path, 'taxonomy.yaml not found — upload it to ROOT'
TAX = yaml.safe_load(open(tax_path))
print('taxonomy:', tax_path)
print('palette:', len(TAX['palette']), '| slots:', list(TAX['slots']))

# (group, value, prompt) in a fixed order
PROMPTS = []
for slot, spec in TAX['slots'].items():
    PROMPTS.append(('slot', slot, spec['prompt']))
for group in ('patterns', 'fits', 'formality', 'seasons', 'materials'):
    for value, p in TAX[group].items():
        PROMPTS.append((group, str(value), p))
for color in TAX['palette']:
    PROMPTS.append(('color_ml', color, f'a garment in the color {color}'))
print(len(PROMPTS), 'prompts')

In [ ]:
import torch
import open_clip

CANDIDATES = [
    ('MobileCLIP-S1', 'datacompdr'),
    ('MobileCLIP2-S0', 'dfndr2b'),
    ('MobileCLIP-S2', 'datacompdr'),
]
model = preprocess = tokenizer = None
MODEL_NAME = PRETRAINED = None
for name, pretrained in CANDIDATES:
    try:
        print('loading', name, pretrained, '...', flush=True)
        model, _, preprocess = open_clip.create_model_and_transforms(name, pretrained=pretrained)
        tokenizer = open_clip.get_tokenizer(name)
        MODEL_NAME, PRETRAINED = name, pretrained
        break
    except Exception as e:
        print('  failed:', type(e).__name__, str(e)[:200])
assert model is not None, 'no MobileCLIP checkpoint loaded'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.eval().to(device)
_size = getattr(model.visual, 'image_size', 256) or 256
IMG_SIZE = int(_size[0]) if isinstance(_size, (tuple, list)) else int(_size)
print('loaded', MODEL_NAME, '| image size', IMG_SIZE, '| device', device)

In [ ]:
@torch.no_grad()
def embed_texts(texts, batch=128):
    out = []
    for i in range(0, len(texts), batch):
        tok = tokenizer(texts[i:i+batch]).to(device)
        e = model.encode_text(tok).float()
        e = e / e.norm(dim=-1, keepdim=True)
        out.append(e.cpu().numpy())
    return np.vstack(out).astype(np.float32)

@torch.no_grad()
def embed_images(pils, batch=64):
    if not pils:
        return np.zeros((0, prompt_emb.shape[1]), dtype=np.float32)
    out = []
    for i in range(0, len(pils), batch):
        tens = torch.stack([preprocess(p) for p in pils[i:i+batch]]).to(device)
        e = model.encode_image(tens).float()
        e = e / e.norm(dim=-1, keepdim=True)
        out.append(e.cpu().numpy())
    return np.vstack(out).astype(np.float32)

P = np.array([p for _, _, p in PROMPTS])
prompt_emb = embed_texts(list(P))
np.save(os.path.join(ART, 'prompt_embeddings.npy'), prompt_emb)
json.dump({
    'model': MODEL_NAME,
    'pretrained': PRETRAINED,
    'image_size': IMG_SIZE,
    'embed_dim': int(prompt_emb.shape[1]),
    'prompts': [f'{g}:{v}' for g, v, _ in PROMPTS],
}, open(os.path.join(ART, 'taxonomy.json'), 'w'), indent=1)
print('prompt embeddings:', prompt_emb.shape)

GROUPS = {}
for i, (g, v, _) in enumerate(PROMPTS):
    GROUPS.setdefault(g, ([], []))
    GROUPS[g][0].append(i)
    GROUPS[g][1].append(v)

def sims_for(embeds, group):
    idx, values = GROUPS[group]
    return embeds @ prompt_emb[idx].T, values

def top1(sims, values):
    return [values[int(i)] for i in sims.argmax(1)]

## 1. Slot accuracy — DeepFashion2 validation

Crops come from the bbox column; labels from `category_id` mapped through `taxonomy.yaml` (`slots.*.categories`).

In [ ]:
df2 = pd.read_csv(os.path.join(RAW, 'deepfashion2', 'validation.csv'))
print(df2.shape, df2.columns.tolist())

CAT2SLOT = {}
for slot, spec in TAX['slots'].items():
    for cid in spec.get('categories') or []:
        CAT2SLOT[int(cid)] = slot
df2['slot'] = df2['category_id'].map(CAT2SLOT)
df2 = df2.dropna(subset=['slot']).copy()
print(df2['slot'].value_counts().to_string())

from PIL import Image

def df2_crop(row):
    p = os.path.join(RAW, 'deepfashion2', 'images', os.path.basename(str(row['path'])))
    if not os.path.exists(p):
        return None
    try:
        im = Image.open(p).convert('RGB')
    except Exception:
        return None
    bb = row['b_box']
    if isinstance(bb, str):
        bb = ast.literal_eval(bb)
    x1, y1, x2, y2 = [float(v) for v in bb]
    W, H = im.size
    x1, y1 = max(0, int(x1)), max(0, int(y1))
    x2, y2 = min(W, int(x2)), min(H, int(y2))
    if x2 - x1 < 8 or y2 - y1 < 8:
        return None
    return im.crop((x1, y1, x2, y2)).resize((256, 256))

N_PER_SLOT = 600
df2_s = (df2.groupby('slot', group_keys=False)
            .apply(lambda g: g.sample(min(len(g), N_PER_SLOT), random_state=0))
            .reset_index(drop=True))
df2_pils, df2_labels = [], []
for _, row in df2_s.iterrows():
    im = df2_crop(row)
    if im is not None:
        df2_pils.append(im); df2_labels.append(row['slot'])
print('crops:', len(df2_pils))

df2_emb = embed_images(df2_pils)
slot_sims, slot_vals = sims_for(df2_emb, 'slot')
slot_pred = top1(slot_sims, slot_vals)
slot_acc = float(np.mean([p == y for p, y in zip(slot_pred, df2_labels)]))
print(f'SLOT accuracy: {slot_acc:.3f}  (gate 0.90, n={len(df2_labels)})')
print('pred:', Counter(slot_pred).most_common())
print('true:', Counter(df2_labels).most_common())

## 2. Pattern + fit accuracy — Fashionpedia attributes

Fashionpedia annotates each item with `attribute_ids`. We keep items with exactly **one** attribute in the target group, then crop the bbox. Images come from the HF parquet mirror, joined on `image_id`.

In [ ]:
import pyarrow.parquet as pq

ANN = os.path.join(RAW, 'fashionpedia', 'annotations', 'val.json')
d = json.load(open(ANN))
attr_name = {a['id']: a['name'].lower().strip() for a in d['attributes']}

PATTERN_MAP = {
    'plain (pattern)': 'solid',
    'check': 'check',
    'floral': 'floral',
    'stripe': 'striped',
}
FIT_MAP = {
    'tight (fit)': 'slim',
    'regular (fit)': 'regular',
    'loose (fit)': 'relaxed',
    'oversized': 'oversized',
}

def group_label(attr_ids, mapping):
    hits = [mapping[attr_name[i]] for i in attr_ids if attr_name.get(i) in mapping]
    return hits[0] if len(set(hits)) == 1 else None

samples = []
for a in d['annotations']:
    aids = a.get('attribute_ids') or []
    pat = group_label(aids, PATTERN_MAP)
    fit = group_label(aids, FIT_MAP)
    if pat or fit:
        samples.append({
            'image_id': a['image_id'], 'bbox': a['bbox'],
            'pattern': pat, 'fit': fit,
        })
print('labeled annotations:', len(samples))
print('pattern:', Counter(s['pattern'] for s in samples if s['pattern']))
print('fit:', Counter(s['fit'] for s in samples if s['fit']))

PQ = os.path.join(RAW, 'fashionpedia', 'hf', 'data', 'val-00000-of-00001-f915eb4a6ec8c97e.parquet')
tbl = pq.read_table(PQ, columns=['image_id', 'image'])
cols = tbl.to_pydict()
id2img = dict(zip(cols['image_id'], cols['image']))
print('parquet images:', len(id2img))

def cell_to_pil(cell):
    if isinstance(cell, dict):
        b = cell.get('bytes')
        if b:
            return Image.open(io.BytesIO(b)).convert('RGB')
        p = cell.get('path')
        if p and os.path.exists(p):
            return Image.open(p).convert('RGB')
    if isinstance(cell, (bytes, bytearray)):
        return Image.open(io.BytesIO(cell)).convert('RGB')
    return None

def fp_crop(s):
    im = cell_to_pil(id2img.get(s['image_id']))
    if im is None:
        return None
    x, y, w, h = [float(v) for v in s['bbox']]
    W, H = im.size
    x1, y1 = max(0, int(x)), max(0, int(y))
    x2, y2 = min(W, int(x + w)), min(H, int(y + h))
    if x2 - x1 < 8 or y2 - y1 < 8:
        return None
    return im.crop((x1, y1, x2, y2)).resize((256, 256))

MAX_PER_CLASS = 400
def balanced(subset, key):
    out = []
    for label, g in pd.DataFrame(subset).groupby(key):
        out.append(g.sample(min(len(g), MAX_PER_CLASS), random_state=0))
    return pd.concat(out).reset_index(drop=True) if out else pd.DataFrame()

pat_s = balanced([s for s in samples if s['pattern']], 'pattern')
fit_s = balanced([s for s in samples if s['fit']], 'fit')
print('pattern samples:', len(pat_s), '| fit samples:', len(fit_s))

# encode each subset once (some crops overlap, cache by image_id+bbox)
cache = {}
def encode_rows(rows):
    pils, keep = [], []
    for i, row in rows.iterrows():
        k = (row['image_id'], tuple(row['bbox']))
        if k not in cache:
            cache[k] = fp_crop(row)
        im = cache[k]
        if im is not None:
            pils.append(im); keep.append(i)
    return embed_images(pils), keep

def eval_subset(rows, key, group, gate, name):
    if len(rows) == 0:
        print(f'{name}: no labeled samples — skipped')
        return None, rows, np.zeros((0, prompt_emb.shape[1]), dtype=np.float32)
    emb, keep = encode_rows(rows)
    rows = rows.loc[keep].reset_index(drop=True)
    if len(rows) == 0:
        print(f'{name}: no decodable crops — skipped')
        return None, rows, emb
    sims, vals = sims_for(emb, group)
    pred = top1(sims, vals)
    acc = float(np.mean([p == y for p, y in zip(pred, rows[key])]))
    print(f'{name} accuracy: {acc:.3f}  (gate {gate}, n={len(rows)})')
    return acc, rows, emb

pat_acc, pat_s, pat_emb = eval_subset(pat_s, 'pattern', 'pattern', 0.80, 'PATTERN')
fit_acc, fit_s, fit_emb = eval_subset(fit_s, 'fit', 'fit', 0.75, 'FIT')

## 3. Color + formality — Fashion Product Images

`baseColour` gives a real color label. `usage` gives a coarse formality band. Product shots have clean backgrounds, so classical CV color extraction can be measured honestly.

In [ ]:
styles = pd.read_csv(os.path.join(RAW, 'fashion-product', 'styles.csv'), on_bad_lines='skip')
print(styles.shape, styles.columns.tolist())
print('usage:', styles['usage'].value_counts().to_dict())

COLOR_ALIASES = {
    'black': 'black', 'white': 'white', 'off white': 'white', 'cream': 'cream',
    'beige': 'tan', 'tan': 'tan', 'khaki': 'tan', 'taupe': 'tan',
    'brown': 'brown', 'coffee brown': 'brown',
    'grey': 'grey', 'gray': 'grey', 'charcoal': 'grey', 'silver': 'grey', 'metallic': 'grey',
    'navy blue': 'navy', 'navy': 'navy',
    'blue': 'blue', 'turquoise blue': 'teal', 'teal': 'teal',
    'green': 'green', 'olive': 'green',
    'red': 'red', 'maroon': 'red', 'burgundy': 'red',
    'rust': 'rust', 'orange': 'orange', 'peach': 'orange',
    'yellow': 'yellow', 'gold': 'yellow',
    'pink': 'pink', 'rose': 'pink',
    'purple': 'purple', 'lavender': 'purple', 'magenta': 'purple',
}
USAGE_BAND = {
    'casual': '1', 'sports': '1', 'travel': '1',
    'smart casual': '3',
    'ethnic': '3',
    'formal': '4', 'party': '4',
}
styles['color'] = styles['baseColour'].astype(str).str.lower().str.strip().map(COLOR_ALIASES)
styles['formality'] = styles['usage'].astype(str).str.lower().str.strip().map(USAGE_BAND)
styles['img'] = styles['id'].astype(str).apply(
    lambda i: os.path.join(RAW, 'fashion-product', 'images', i + '.jpg'))

prod = styles.dropna(subset=['img']).copy()
prod = prod[prod['img'].apply(os.path.exists)]
print('images available:', len(prod))

def product_slot(row):
    at = str(row.get('articleType', '')).lower()
    sub = str(row.get('subCategory', '')).lower()
    master = str(row.get('masterCategory', '')).lower()
    if 'dress' in at:
        return 'dress'
    if master == 'footwear':
        return 'shoes'
    if master == 'accessories':
        return 'accessory'
    if 'outerwear' in sub or 'jacket' in at or 'coat' in at or 'blazer' in at:
        return 'outerwear'
    if 'topwear' in sub:
        return 'top'
    if 'bottomwear' in sub:
        return 'bottom'
    return None

prod['slot'] = prod.apply(product_slot, axis=1)
print('product slot labels:', prod['slot'].value_counts().to_dict())

def balanced_any(df, key, n):
    parts = []
    for _, g in df.dropna(subset=[key]).groupby(key):
        parts.append(g.sample(min(len(g), n), random_state=0))
    return pd.concat(parts).reset_index(drop=True) if parts else pd.DataFrame()

prod_s = balanced_any(prod, 'color', 250)
prod_s = prod_s.drop_duplicates(subset=['id'])
prod_pils = [Image.open(p).convert('RGB') for p in prod_s['img']]
prod_emb = embed_images(prod_pils)
print('product samples:', len(prod_s))

color_sims, color_vals = sims_for(prod_emb, 'color_ml')
color_pred = top1(color_sims, color_vals)
mask = prod_s['color'].notna().values
color_ml_acc = float(np.mean([p == y for p, y in zip(np.array(color_pred)[mask], prod_s.loc[mask, 'color'])]))
print(f'COLOR (zero-shot) accuracy: {color_ml_acc:.3f}  (diagnostic, n={mask.sum()})')

In [ ]:
import cv2
from sklearn.cluster import MiniBatchKMeans

palette_rgb = {k: np.array(v, dtype=np.uint8) for k, v in TAX['palette'].items()}
palette_lab = {}
for k, rgb in palette_rgb.items():
    lab = cv2.cvtColor(rgb.reshape(1, 1, 3), cv2.COLOR_RGB2LAB).astype(np.float32)
    palette_lab[k] = lab.ravel()

def dominant_color(pil, k=3):
    img = np.array(pil.resize((64, 64)))
    h, w = img.shape[:2]
    img = img[int(h * .2):int(h * .8), int(w * .2):int(w * .8)]
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB).reshape(-1, 3).astype(np.float32)
    km = MiniBatchKMeans(n_clusters=k, n_init=3, random_state=0).fit(lab)
    counts = np.bincount(km.labels_)
    names = [min(palette_lab, key=lambda n: float(np.linalg.norm(c - palette_lab[n])))
             for c in km.cluster_centers_]
    order = counts.argsort()[::-1]
    for idx in order:
        # skip a white cluster that is probably the studio background
        if names[idx] == 'white' and counts[idx] < 0.9 * len(lab):
            continue
        return names[idx]
    return names[order[0]]

cv_pred = [dominant_color(p) for p in prod_pils]
color_cv_acc = float(np.mean([p == y for p, y in zip(np.array(cv_pred)[mask], prod_s.loc[mask, 'color'])]))
print(f'COLOR (classical CV) accuracy: {color_cv_acc:.3f}  (gate 0.85, n={mask.sum()})')

fm_sims, fm_vals = sims_for(prod_emb, 'formality')
fm_pred = top1(fm_sims, fm_vals)
fm_mask = prod_s['formality'].notna().values
exact = float(np.mean([p == y for p, y in zip(np.array(fm_pred)[fm_mask], prod_s.loc[fm_mask, 'formality'])]))
within1 = float(np.mean([abs(int(p) - int(y)) <= 1 for p, y in zip(np.array(fm_pred)[fm_mask], prod_s.loc[fm_mask, 'formality'])]))
print(f'FORMALITY exact: {exact:.3f}  | within ±1: {within1:.3f}  (gate 0.70 exact, n={fm_mask.sum()})')

## 4. Visual grid — material / season (human check)

No public label set for these; eyeball the grid. If a category is consistently wrong, add a prompt or run a probe in 02.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def save_grid(pils, titles, path, ncols=6, n=24):
    idx = np.linspace(0, len(pils) - 1, min(n, len(pils))).astype(int)
    rows = int(np.ceil(len(idx) / ncols))
    fig, axes = plt.subplots(rows, ncols, figsize=(ncols * 2.2, rows * 2.4))
    axes = np.array(axes).flatten()
    for j, i in enumerate(idx):
        axes[j].imshow(pils[i]); axes[j].set_title(str(titles[i])[:26], fontsize=7); axes[j].axis('off')
    for j in range(len(idx), len(axes)):
        axes[j].axis('off')
    plt.tight_layout(); plt.savefig(path, dpi=90); plt.close()

save_grid(df2_pils, [f'{p} / {y}' for p, y in zip(slot_pred, df2_labels)],
          os.path.join(ART, 'visual_grid_slots.png'))
print('wrote visual_grid_slots.png (title = predicted / true)')

## 5. Export the image tower → ONNX (the app ships ONNX Runtime)

We tried TFLite (`onnx2tf`) and int8 dynamic quantization. Both broke parity on FastViT (cosine -0.12 and 0.11 vs PyTorch). The fp32 ONNX is exact (cosine 1.0), so the app ships `onnxruntime-react-native`. 88 MB is acceptable for the MVP; static int8 quantization with a calibration set is the later optimization.

In [ ]:
import torch
model.eval().cpu()
visual = model.visual.eval()
onnx_path = os.path.join(ART, 'mobileclip_s1_image.onnx')
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
torch.onnx.export(
    visual, (dummy,), onnx_path,
    opset_version=17, do_constant_folding=True,
    input_names=['pixels'], output_names=['features'])
# fold external weights into the .onnx file so the bundle is self-contained
import onnx
from onnx.external_data_helper import convert_model_from_external_data
_m = onnx.load(onnx_path)
convert_model_from_external_data(_m)
onnx.save_model(_m, onnx_path, save_as_external_data=False)
data_file = onnx_path + '.data'
if os.path.exists(data_file):
    os.remove(data_file)
print('onnx:', onnx_path, f'{os.path.getsize(onnx_path)/1e6:.1f} MB')

with torch.no_grad():
    torch_out = visual(dummy).cpu().numpy().ravel()
torch_out = torch_out / np.linalg.norm(torch_out)

# parity: PyTorch vs ONNX Runtime (the runtime the app ships)
import onnxruntime as ort
sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
idx = np.random.default_rng(0).choice(len(df2_pils), min(20, len(df2_pils)), replace=False)
min_cos = 1.0
for i in idx:
    t = preprocess(df2_pils[i]).numpy()[None].astype(np.float32)
    a = sess.run(None, {sess.get_inputs()[0].name: t})[0].ravel()
    a = a / (np.linalg.norm(a) + 1e-9)
    b = df2_emb[i].astype(np.float32)
    b = b / (np.linalg.norm(b) + 1e-9)
    min_cos = min(min_cos, float(np.dot(a, b)))
print(f'min cosine PyTorch-vs-ONNX: {min_cos:.6f}  (need > 0.999)')
assert min_cos > 0.999, 'ONNX parity failed — do not ship this export'


## 6. Probe data + eval report + bundle

`probe_data.npz` is what notebook 02 trains on. Only attributes that fail a gate need a probe.

In [ ]:
prod_slot_mask = prod_s['slot'].notna().values
if prod_slot_mask.sum() > 0:
    X_slot = np.vstack([df2_emb, prod_emb[prod_slot_mask]])
    y_slot = np.concatenate([np.array(df2_labels), prod_s.loc[prod_slot_mask, 'slot'].values])
else:
    X_slot, y_slot = df2_emb, np.array(df2_labels)
print('slot probe data:', X_slot.shape, 'classes:', sorted(set(y_slot)))

np.savez_compressed(
    os.path.join(ART, 'probe_data.npz'),
    X_slot=X_slot, y_slot=y_slot,
    X_pattern=pat_emb, y_pattern=pat_s['pattern'].values,
    X_fit=fit_emb, y_fit=fit_s['fit'].values,
    X_formality=prod_emb[fm_mask], y_formality=prod_s.loc[fm_mask, 'formality'].values,
    X_color=prod_emb[mask], y_color=prod_s.loc[mask, 'color'].values,
)
print('probe_data.npz written')

def gated(acc, gate, n, note=None):
    if acc is None:
        return {'acc': None, 'gate': gate, 'n': n, 'note': note or 'no matching public eval split'}
    return {'acc': acc, 'gate': gate, 'n': n}

report = {
    'model': MODEL_NAME,
    'image_size': IMG_SIZE,
    'slot': gated(slot_acc, 0.90, len(df2_labels)),
    'pattern': gated(pat_acc, 0.80, len(pat_s),
                     'Fashionpedia public mirrors ship attributes and images with disjoint id remaps; '
                     'pattern stays zero-shot + user corrections for MVP'),
    'fit': gated(fit_acc, 0.75, len(fit_s),
                 'same Fashionpedia id-remap issue; fit stays zero-shot + user corrections for MVP'),
    'color_cv': gated(color_cv_acc, 0.85, int(mask.sum())),
    'color_ml': {'acc': color_ml_acc, 'gate': None, 'n': int(mask.sum())},
    'formality': {'acc': exact, 'within1': within1, 'gate': 0.70, 'n': int(fm_mask.sum())},
}
json.dump(report, open(os.path.join(ART, 'eval_report.json'), 'w'), indent=1)

lines = [f'# StyleMe vision eval report', '', f'model: **{MODEL_NAME}** ({PRETRAINED}), {IMG_SIZE}px', '']
for name, r in report.items():
    if not isinstance(r, dict):
        continue
    gate = r.get('gate')
    acc = r.get('acc')
    if acc is None:
        lines.append(f'- **{name}**: skipped — {r.get("note")}')
    elif gate is None:
        lines.append(f'- **{name}**: {acc:.3f} (diagnostic, n={r["n"]})')
    else:
        ok = acc >= gate
        extra = f', within±1 {r["within1"]:.3f}' if 'within1' in r else ''
        lines.append(f'- **{name}**: {acc:.3f} vs gate {gate} — {"PASS" if ok else "**FAIL**"}{extra} (n={r["n"]})')
lines += ['', 'Failures: run `02_probe_finetune_and_export.ipynb` for those groups, then re-bundle.']
text = '\n'.join(lines)
open(os.path.join(ART, 'eval_report.md'), 'w').write(text)
print(text)

bundle = [os.path.join(ART, f) for f in [
    'prompt_embeddings.npy', 'taxonomy.json', 'eval_report.md', 'eval_report.json',
    'probe_data.npz', 'visual_grid_slots.png']]
bundle += [p for p in [onnx_path] if p and os.path.exists(p)]
zip_path = os.path.join(ART, 'styleme_vision_v1.zip')
if os.path.exists(zip_path):
    os.remove(zip_path)
subprocess.run(['zip', '-q', '-j', zip_path] + bundle, check=True)
print('bundle:', zip_path, f'{os.path.getsize(zip_path)/1e6:.1f} MB')

## Next

- **All gates pass** → copy `styleme_vision_v1.zip` into the app build (see `apps/mobile/src/vision/README.md`).
- **Any gate fails** → run **02_probe_finetune_and_export.ipynb**, set `PROBE_GROUPS` to the failed attributes, then re-bundle.
- Notebook **03** is independent: it calibrates the engine's compatibility weights from real outfits.